# M3L3 E17 — Vendedores especializados con RAG
### Módulo 3 · Lecture 3 · Sistemas Multiagente Avanzados

**Caso terminado:** arquitectura de agentes de ventas con RAG real sobre catálogos especializados.

## ¿Qué vas a ver en este ejercicio?
- Un router LLM que identifica la línea de producto correcta.
- Agentes de ventas especializados que recuperan contexto del catálogo antes de responder (RAG).
- Un agente fallback para consultas fuera de las líneas de producto.

## Arquitectura del sistema

> **RAG en ventas:** el agente recupera información del catálogo antes de generar la respuesta. El LLM actúa como vendedor experto que conoce los productos en profundidad.

```
START
  |
  v
router_sales_node
  |                        |
  v                        v
software_sales_agent   hardware_sales_agent
  |                        |
  +----------+-------------+
             |
        fallback_agent
             |
            END
```

| Agente | Especialidad | Knowledge base |
|---|---|---|
| `software_sales_agent` | SaaS, licencias, suscripciones | `software_kb` |
| `hardware_sales_agent` | Servidores, laptops, periféricos | `hardware_kb` |
| `fallback_agent` | Consultas fuera de catálogo | Respuesta genérica |

## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

## Paso 2 — Instalar LangGraph

In [ ]:
!pip install langgraph -q

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph listo.")

## Sección 1 — Knowledge bases de ventas

> **Knowledge base simulada:** diccionario in-memory que representa el catálogo de productos. En producción esto sería un vector store o base de datos.

In [ ]:
software_kb = [
    "Plan Starter: $29/mes, hasta 5 usuarios, 10 GB almacenamiento, soporte por email.",
    "Plan Pro: $99/mes, hasta 25 usuarios, 100 GB almacenamiento, soporte prioritario 24/7.",
    "Plan Enterprise: precio a medida, usuarios ilimitados, almacenamiento ilimitado, SLA 99.99%, gerente de cuenta dedicado.",
    "Licencias anuales incluyen 20% de descuento sobre el precio mensual.",
    "Integraciones disponibles: Slack, Salesforce, HubSpot, Zapier, API REST.",
    "Período de prueba gratuito: 14 días sin tarjeta de crédito.",
    "Migración de datos incluida gratis en planes Pro y Enterprise.",
]

hardware_kb = [
    "Servidor RackPro X1: Intel Xeon 32 cores, 256GB RAM, 4TB NVMe, $8.500 + IVA.",
    "Servidor RackPro X2: AMD EPYC 64 cores, 512GB RAM, 8TB NVMe RAID, $15.000 + IVA.",
    "Laptop WorkForce 14: Core i7 13th Gen, 16GB RAM, 512GB SSD, pantalla 14\" FHD, $1.200.",
    "Laptop WorkForce 16 Pro: Core i9, 32GB RAM, 1TB SSD, pantalla 16\" 4K, $2.100.",
    "Monitor UltraView 27\": 4K IPS, 144Hz, HDR400, compatible USB-C, $650.",
    "Garantía extendida: 3 años on-site disponible para todos los equipos, $200 adicionales.",
    "Envío gratis en compras superiores a $500. Entrega en 5 días hábiles.",
]

print(f"software_kb: {len(software_kb)} entradas")
print(f"hardware_kb: {len(hardware_kb)} entradas")

## Sección 2 — State del sistema de ventas

> **SalesState:** el estado acumula la clasificación del router y la respuesta del agente especializado.

In [ ]:
class SalesState(TypedDict):
    query: str
    product_line: str   # "software" | "hardware" | "unknown"
    reason: str
    response: str

## Sección 3 — Nodos del sistema

> **Router de ventas:** el LLM analiza la intención del cliente y determina qué línea de producto corresponde. Los agentes especialistas usan RAG para dar respuestas con precios y detalles reales del catálogo.

In [ ]:
def router_sales_node(state: SalesState) -> dict:
    prompt = (
        "Sos el router de un sistema de ventas tecnológicas.\n"
        "Clasificá la consulta del cliente en exactamente una categoría:\n"
        "- 'software': consultas sobre SaaS, planes, licencias, suscripciones, integraciones\n"
        "- 'hardware': consultas sobre servidores, laptops, monitores, periféricos, garantías\n"
        "- 'unknown': consultas que no corresponden a ninguna línea de producto\n\n"
        "Respondé con JSON: {\"product_line\": \"...\", \"reason\": \"...\"}\n"
        "Solo product_line y reason, sin markdown.\n\n"
        f"Consulta del cliente: {state['query']}"
    )
    response = llm.invoke(prompt)
    import json, re
    text = response.content.strip()
    text = re.sub(r"```[\w]*\n?", "", text).strip()
    try:
        data = json.loads(text)
        product_line = data.get("product_line", "unknown").lower()
        reason = data.get("reason", "")
    except Exception:
        product_line = "unknown"
        reason = text
    if product_line not in ("software", "hardware"):
        product_line = "unknown"
    return {"product_line": product_line, "reason": reason}


def software_sales_agent(state: SalesState) -> dict:
    context = "\n".join(software_kb)
    response = llm.invoke(
        "Sos un agente de ventas especializado en soluciones de software SaaS.\n"
        "Tu objetivo es ayudar al cliente a encontrar el plan o solución que mejor se adapte a sus necesidades.\n"
        "Usá el siguiente catálogo para dar información precisa y recomendaciones.\n\n"
        f"Catálogo de software:\n{context}\n\n"
        f"Consulta del cliente: {state['query']}\n\n"
        "Respondé de forma amigable y profesional en español. Incluí precios si es relevante."
    )
    return {"response": response.content.strip()}


def hardware_sales_agent(state: SalesState) -> dict:
    context = "\n".join(hardware_kb)
    response = llm.invoke(
        "Sos un agente de ventas especializado en hardware tecnológico.\n"
        "Tu objetivo es ayudar al cliente a encontrar el equipo que mejor se adapte a sus necesidades.\n"
        "Usá el siguiente catálogo para dar información precisa con especificaciones y precios.\n\n"
        f"Catálogo de hardware:\n{context}\n\n"
        f"Consulta del cliente: {state['query']}\n\n"
        "Respondé de forma amigable y profesional en español. Incluí especificaciones técnicas relevantes."
    )
    return {"response": response.content.strip()}


def fallback_agent(state: SalesState) -> dict:
    return {
        "response": (
            "Gracias por contactarnos. Nuestro equipo de ventas se especializa en soluciones "
            "de software SaaS y hardware tecnológico. ¿Podés contarme más sobre qué tipo de "
            "solución tecnológica estás buscando? Estaré encantado de ayudarte."
        )
    }


def sales_router(state: SalesState) -> str:
    return {
        "software": "software_sales_agent",
        "hardware": "hardware_sales_agent",
    }.get(state["product_line"], "fallback_agent")


print("Nodos definidos.")

## Sección 4 — Compilar el grafo

In [ ]:
graph = StateGraph(SalesState)

graph.add_node("router_sales_node",    router_sales_node)
graph.add_node("software_sales_agent", software_sales_agent)
graph.add_node("hardware_sales_agent", hardware_sales_agent)
graph.add_node("fallback_agent",       fallback_agent)

graph.add_edge(START, "router_sales_node")
graph.add_conditional_edges(
    "router_sales_node",
    sales_router,
    {
        "software_sales_agent": "software_sales_agent",
        "hardware_sales_agent": "hardware_sales_agent",
        "fallback_agent": "fallback_agent",
    },
)
for node in ["software_sales_agent", "hardware_sales_agent", "fallback_agent"]:
    graph.add_edge(node, END)

app = graph.compile()
print("Grafo compilado.")

## Demo — Consultas de clientes

El sistema enruta cada consulta al agente correcto y genera una respuesta con información real del catálogo.

In [ ]:
EMPTY = {"query": "", "product_line": "", "reason": "", "response": ""}

queries = [
    "¿Qué plan tienen para una empresa de 15 personas?",
    "Necesito un servidor para montar un cluster de base de datos",
    "¿Tienen laptops para diseñadores gráficos?",
    "Quiero integrar su plataforma con Salesforce",
    "¿Hacen catering para eventos corporativos?",
]

for q in queries:
    r = app.invoke({**EMPTY, "query": q})
    print(f"\nConsulta: {q}")
    print(f"Línea:    {r['product_line']} | Razón: {r['reason'][:60]}")
    print(f"Respuesta: {r['response'][:120]}...")
    print("-" * 70)

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    empty = {"query": "", "product_line": "", "reason": "", "response": ""}

    r1 = app.invoke({**empty, "query": "¿cuánto cuesta el plan Enterprise con usuarios ilimitados?"})
    assert r1["product_line"] == "software", f"esperaba software: {r1['product_line']}"
    assert len(r1["response"]) > 10, "respuesta vacía"

    r2 = app.invoke({**empty, "query": "quiero comprar 10 laptops para mi equipo de desarrollo"})
    assert r2["product_line"] == "hardware", f"esperaba hardware: {r2['product_line']}"
    assert len(r2["response"]) > 10, "respuesta vacía"

    r3 = app.invoke({**empty, "query": "¿dan clases de cocina?"})
    assert r3["product_line"] == "unknown", f"esperaba unknown: {r3['product_line']}"

    print("Checks E17 OK")

run_checks()

## ¿Qué viste en este caso?

- El router LLM clasifica la intención de compra en la línea de producto correcta.
- Los agentes especialistas usan RAG: recuperan contexto del catálogo antes de generar la respuesta.
- El fallback captura consultas fuera de las líneas de producto disponibles.

| Concepto | Implementación en E17 |
|---|---|
| Router LLM | `router_sales_node` devuelve JSON `{product_line, reason}` |
| RAG | `"\n".join(kb)` → string de contexto inyectado en el prompt del agente |
| Edge condicional | `sales_router()` mapea `product_line` → nombre del nodo |
| Fallback | `fallback_agent` para cualquier `product_line` no reconocida |

## Próximo ejercicio

En **E18** vas a ver un sistema SaaS con 3 departamentos especializados: producto, soporte técnico y facturación.